# EDA — Interactive Exploration

Thin wrapper around `analysis/eda.py`. Use this notebook to poke at the datasets interactively — each function writes the same outputs as the script, but you can call them on whichever subset you like and inspect the DataFrames / plots inline.

For the reproducible run that produces the final report artefacts, run `python analysis/eda.py` from the repo root instead.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

# Make sure we can import from the repo root
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis.eda import (
    compute_basic_stats, summarise_stats,
    run_stationarity, compute_seasonality_strength, compute_mean_acf,
    plot_length_distribution, plot_scale_distribution, plot_value_distribution,
    plot_sample_series, plot_mean_acf, plot_seasonality_strength, plot_zero_fraction,
)
from data_prep.m4_prep import load_m4_monthly
from data_prep.m5_prep import load_m5
from data_prep.traffic_prep import load_traffic
from config import M4_CONFIG, M5_CONFIG, TRAFFIC_CONFIG

## 1. Load a dataset

Start with the sampled subset — it matches what the models actually see and loads in seconds. Switch `n_series=None` to inspect the full dataset (much slower for M5).

In [ ]:
# Pick one — try all three eventually
df_train, df_test = load_m4_monthly(n_series=500)
cfg = M4_CONFIG

# df_train, df_test = load_m5(n_series=200);       cfg = M5_CONFIG
# df_train, df_test = load_traffic(n_series=50);   cfg = TRAFFIC_CONFIG

df = pd.concat([df_train, df_test], ignore_index=True)
print(f"{cfg['name']}: {df['unique_id'].nunique()} series, {len(df):,} rows")
df.head()

## 2. Basic per-series statistics

In [ ]:
stats = compute_basic_stats(df)
print(f'{len(stats)} series')
stats.describe()

In [ ]:
summary = summarise_stats(stats)
import json; print(json.dumps(summary, indent=2, default=str))

## 3. Stationarity (ADF + KPSS) on a random sample of 100 series

Quick sanity check that matches what the full script produces. Bump to 500 for publication numbers.

In [ ]:
stationarity = run_stationarity(df, n_sample=100)
stationarity

## 4. Seasonality strength and mean ACF

In [ ]:
fs = compute_seasonality_strength(df, season_length=cfg['season_length'], n_sample=100)
print(f'STL Fs — median {np.median(fs):.3f}, p25 {np.percentile(fs, 25):.3f}, p75 {np.percentile(fs, 75):.3f}')

max_lag = min(3 * cfg['season_length'], 100)
acf_vals = compute_mean_acf(df, max_lag=max_lag, n_sample=100)
print(f'ACF lag-1 = {acf_vals[1]:.3f}, ACF lag-seasonal = {acf_vals[cfg["season_length"]]:.3f}')

## 5. Inline plots

Each of these writes a PNG to `/tmp` and also returns a matplotlib Figure you can inspect inline.

In [ ]:
from pathlib import Path
tmp = Path('/tmp/eda_explore')
tmp.mkdir(exist_ok=True)

name = cfg['name']
plot_length_distribution(stats, tmp / 'length.png', name);       plt.show()
plot_scale_distribution (stats, tmp / 'scale.png',  name);       plt.show()
plot_value_distribution (df,    tmp / 'values.png', name);       plt.show()
plot_sample_series      (df,    tmp / 'samples.png', name);      plt.show()
plot_mean_acf           (acf_vals, cfg['season_length'], tmp / 'acf.png', name);  plt.show()
plot_seasonality_strength(fs,   tmp / 'fs.png',    name);        plt.show()
plot_zero_fraction      (stats, tmp / 'zeros.png', name);        plt.show()

## 6. Ad-hoc: inspect the most-zero series in M5

Concrete example of how to use the per-series stats table to find problem cases. Swap in whatever filter is interesting.

In [ ]:
worst_zero = stats.nlargest(5, 'pct_zero')
worst_zero

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=False)
for ax, uid in zip(axes, worst_zero['unique_id']):
    sub = df[df['unique_id'] == uid].sort_values('ds')
    ax.plot(sub['ds'], sub['y'], linewidth=0.6)
    ax.set_title(f'{uid}  ({worst_zero.set_index("unique_id").loc[uid, "pct_zero"]:.1f}% zero)', fontsize=9)
plt.tight_layout()
plt.show()